# 02 - Araç Tespiti Baseline (Pretrained YOLOv8)

COCO pretrained YOLOv8 modellerinin İstanbul trafik videolarındaki performansı.

## Deneyler
- YOLOv8n, YOLOv8s, YOLOv8m karşılaştırması
- Confidence threshold optimizasyonu
- FPS ve doğruluk trade-off analizi

In [ ]:
# !pip install ultralytics opencv-python matplotlib

from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path

In [ ]:
# Test videosu
VIDEO_PATH = 'data/videos/test/sample.mp4'
VEHICLE_CLASSES = [2, 3, 5, 7]  # car, motorcycle, bus, truck

# Modeller
MODELS = {
    'YOLOv8n': 'yolov8n.pt',
    'YOLOv8s': 'yolov8s.pt',
    'YOLOv8m': 'yolov8m.pt',
}

In [ ]:
def evaluate_model(model_path, video_path, num_frames=300, conf=0.35):
    """Model performansını ölç."""
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    
    frame_times = []
    detection_counts = []
    
    for i in range(num_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        t0 = time.time()
        results = model.predict(frame, conf=conf, classes=VEHICLE_CLASSES, verbose=False)
        t1 = time.time()
        
        frame_times.append(t1 - t0)
        n_det = len(results[0].boxes) if results[0].boxes is not None else 0
        detection_counts.append(n_det)
    
    cap.release()
    
    return {
        'avg_fps': 1.0 / np.mean(frame_times),
        'avg_detections': np.mean(detection_counts),
        'total_detections': sum(detection_counts),
        'frame_times': frame_times,
    }

In [ ]:
# Her modeli değerlendir
results = {}
for name, path in MODELS.items():
    print(f'Değerlendiriliyor: {name}')
    results[name] = evaluate_model(path, VIDEO_PATH)
    print(f'  FPS: {results[name]["avg_fps"]:.1f}, '
          f'Ort. tespit: {results[name]["avg_detections"]:.1f}')

In [ ]:
# FPS karşılaştırma
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(results.keys())
fps_values = [results[m]['avg_fps'] for m in model_names]
det_values = [results[m]['avg_detections'] for m in model_names]

axes[0].bar(model_names, fps_values, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[0].set_title('Ortalama FPS')
axes[0].set_ylabel('FPS')

axes[1].bar(model_names, det_values, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[1].set_title('Ortalama Tespit Sayısı / Kare')
axes[1].set_ylabel('Tespit Sayısı')

plt.tight_layout()
plt.savefig('results/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Görsel sonuçlar — örnek kare
model = YOLO('yolov8s.pt')
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 100)
ret, frame = cap.read()
cap.release()

results = model.predict(frame, conf=0.35, classes=VEHICLE_CLASSES)
annotated = results[0].plot()

plt.figure(figsize=(15, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title('YOLOv8s Pretrained — Araç Tespiti')
plt.axis('off')
plt.savefig('results/detection_sample.png', dpi=150, bbox_inches='tight')
plt.show()